# 01 - Exploratory Data Analysis (EDA)

## 0. Imports and upload

In [ ]:
                                                     
                                          

from google.colab import files
uploaded = files.upload()
input_file = next(iter(uploaded))

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew, kurtosis, pearsonr, spearmanr
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import TimeSeriesSplit, GroupKFold, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import SelectKBest, mutual_info_regression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

OUT_DIR = "pm25_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

df_raw = pd.read_csv(input_file)
print("Original shape:", df_raw.shape)
print("Original columns:", df_raw.columns.tolist())
display(df_raw.head())

## 1. Standardize fields and validate the dataset

In [ ]:
                                                                                            
ALIASES = {
    "PM2.5": "pm25", "PM25": "pm25", "PM_25": "pm25",
    "station_id": "id", "station": "id", "station_name": "sensor_name",
    "lat": "latitude", "lon": "longitude", "lng": "longitude",
    "temperature": "temperature_celsius", "temp": "temperature_celsius",
    "pressure": "pressure_mb", "wind_u_component": "wind_u", "wind_v_component": "wind_v",
    "no2": "NO2", "co": "CO", "o3": "O3"
}
df = df_raw.rename(columns={c: ALIASES[c] for c in df_raw.columns if c in ALIASES}).copy()

required_core = ["date", "id", "pm25"]
missing_core = [c for c in required_core if c not in df.columns]
if missing_core:
    raise ValueError(f"Missing required columns: {missing_core}. Check the CSV headers and ALIASES.")

df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
if "datetime_utc" in df.columns:
    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], errors="coerce")

numeric_candidates = [
    "latitude", "longitude", "temperature_celsius", "pressure_mb",
    "wind_u", "wind_v", "NO2", "CO", "O3", "AOD", "pm25"
]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

                                                                                                
df["id"] = df["id"].astype("string")
if "sensor_name" in df.columns:
    df["sensor_name"] = df["sensor_name"].astype("string")

print("Date range before PM2.5 filtering:", df["date"].min(), "to", df["date"].max())
print("Rows before cleaning:", len(df))
print("Stations before cleaning:", df["id"].nunique(dropna=True))
display(df.dtypes.to_frame("dtype"))

## 2. Mandatory PM2.5 cleaning - before EDA and splitting

In [ ]:
n_initial = len(df)
missing_pm25 = int(df["pm25"].isna().sum())
invalid_low = int((df["pm25"].notna() & (df["pm25"] <= 0)).sum())
invalid_high = int((df["pm25"].notna() & (df["pm25"] >= 500)).sum())
invalid_date = int(df["date"].isna().sum())
missing_station = int(df["id"].isna().sum())

                                                          
df = df[df["pm25"].notna() & df["pm25"].gt(0) & df["pm25"].lt(500)].copy()
df = df[df["date"].notna() & df["id"].notna()].copy()
df = df.sort_values(["id", "date"]).reset_index(drop=True)

cleaning_counts = pd.DataFrame({
    "reason": ["Missing PM2.5", "PM2.5 <= 0", "PM2.5 >= 500",
               "Invalid/missing date", "Missing station ID", "Rows retained"],
    "count": [missing_pm25, invalid_low, invalid_high, invalid_date,
              missing_station, len(df)]
})
display(cleaning_counts)
cleaning_counts.to_csv(f"{OUT_DIR}/PM25_cleaning_counts.csv", index=False)

print("Cleaned shape:", df.shape)
print("Cleaned date coverage:", df.date.min(), "to", df.date.max())
print("Stations:", df.id.nunique())
assert df["pm25"].gt(0).all() and df["pm25"].lt(500).all()

## 3. Dataset description, units, temporal and spatial coverage

In [ ]:
variable_description = pd.DataFrame([
    ("pm25", "Target: fine particulate matter concentration", "µg/m³"),
    ("AOD", "Aerosol optical depth", "dimensionless (confirm product scale)"),
    ("NO2", "Sentinel-5P nitrogen dioxide column", "dataset/product-specific; verify source metadata"),
    ("CO", "Sentinel-5P carbon monoxide column", "dataset/product-specific; verify source metadata"),
    ("O3", "Sentinel-5P ozone column", "dataset/product-specific; verify source metadata"),
    ("temperature_celsius", "Air temperature", "°C"),
    ("pressure_mb", "Atmospheric pressure", "mb / hPa"),
    ("wind_u", "East-west wind component", "source-dependent units; verify metadata"),
    ("wind_v", "North-south wind component", "source-dependent units; verify metadata"),
    ("latitude", "Station latitude", "degrees"),
    ("longitude", "Station longitude", "degrees"),
    ("id", "Station identifier", "categorical identifier"),
    ("sensor_name", "Station/sensor label", "categorical identifier")
], columns=["variable", "description", "unit"])
display(variable_description)
variable_description.to_csv(f"{OUT_DIR}/PM25_variable_dictionary.csv", index=False)

print("Rows:", len(df))
print("Stations:", df["id"].nunique())
print("Temporal coverage:", df["date"].min(), "to", df["date"].max())
if {"latitude", "longitude"}.issubset(df.columns):
    print("Latitude range:", df.latitude.min(), "to", df.latitude.max())
    print("Longitude range:", df.longitude.min(), "to", df.longitude.max())
    print("Unique coordinate pairs:", df[["latitude", "longitude"]].drop_duplicates().shape[0])
print("Years:", sorted(df.date.dt.year.unique().tolist()))
display(df[["id"]].value_counts().rename("records_per_station").describe().to_frame())

## 4. Duplicate inspection

In [ ]:
duplicate_all = int(df.duplicated(keep="first").sum())
key_fields = [c for c in ["date", "latitude", "longitude", "id"] if c in df.columns]
duplicate_station_date = int(df.duplicated(["date", "id"], keep="first").sum())
duplicate_location_date = int(df.duplicated(key_fields, keep="first").sum()) if len(key_fields) >= 3 else np.nan

duplicate_report = pd.DataFrame({
    "duplicate_definition": ["All columns", "Date + station ID", "Date + latitude + longitude + station ID"],
    "duplicate_rows": [duplicate_all, duplicate_station_date, duplicate_location_date]
})
display(duplicate_report)
duplicate_report.to_csv(f"{OUT_DIR}/PM25_duplicate_report.csv", index=False)

                                                                                                          
display(df[df.duplicated(["date", "id"], keep=False)].head(20))

## 5. Numerical summaries and categorical identifier counts

In [ ]:
numeric_cols = [c for c in [
    "pm25", "AOD", "NO2", "CO", "O3", "temperature_celsius",
    "pressure_mb", "wind_u", "wind_v", "latitude", "longitude"
] if c in df.columns]

summary = df[numeric_cols].describe(percentiles=[.25, .5, .75]).T
summary = summary.rename(columns={"25%":"q25", "50%":"median", "75%":"q75"})
summary["skewness"] = df[numeric_cols].skew()
summary["kurtosis_excess"] = df[numeric_cols].kurtosis()
display(summary)
summary.to_csv(f"{OUT_DIR}/Exploratory_Data.csv", encoding="utf-8")

for ident in [c for c in ["id", "sensor_name"] if c in df.columns]:
    counts = df[ident].value_counts(dropna=False).rename_axis(ident).reset_index(name="count")
    print(f"Category counts: {ident} (number of categories={df[ident].nunique(dropna=False)})")
    display(counts.head(30))
    counts.to_csv(f"{OUT_DIR}/counts_{ident}.csv", index=False)

## 6. Missingness - overall, by month and by station

In [ ]:
missing_overall = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100)
}).sort_values("missing_percent", ascending=False)
display(missing_overall)
missing_overall.to_csv(f"{OUT_DIR}/PM25_missingness_overall.csv")

satellite_cols = [c for c in ["NO2", "CO", "O3"] if c in df.columns]
if satellite_cols:
    monthly_station_missing = (
        df.assign(month=df["date"].dt.to_period("M").astype(str))
          .groupby(["id", "month"])[satellite_cols]
          .agg(lambda x: x.isna().mean() * 100)
          .reset_index()
    )
    display(monthly_station_missing.head(20))
    monthly_station_missing.to_csv(f"{OUT_DIR}/PM25_missingness_NO2_CO_O3_by_month_station.csv", index=False)

## 7. Histograms, boxplots and categorical plots

In [ ]:
eda_vars = [c for c in [
    "pm25", "AOD", "NO2", "CO", "O3", "temperature_celsius",
    "pressure_mb", "wind_u", "wind_v"
] if c in df.columns]

for col in eda_vars:
    values = df[col].dropna()
    if values.empty:
        print(f"{col}: no observed values")
        continue
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].hist(values, bins=40, edgecolor="black")
    axes[0].set_title(f"Histogram — {col}")
    axes[0].set_xlabel(col)
    axes[1].boxplot(values, vert=False, showfliers=True)
    axes[1].set_title(f"Boxplot — {col}")
    axes[1].set_xlabel(col)
    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"distribution_{col}.png"), dpi=200, bbox_inches="tight")
    plt.show()

                                                                                                  
station_counts = df["id"].value_counts().head(20)
plt.figure(figsize=(12, 5))
station_counts.sort_values().plot(kind="barh")
plt.title("20 stations with the most observations")
plt.xlabel("Number of observations")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "station_observation_counts_top20.png"), dpi=200, bbox_inches="tight")
plt.show()

top_stations = station_counts.index
plt.figure(figsize=(13, 5))
sns.boxplot(data=df[df.id.isin(top_stations)], x="id", y="pm25")
plt.xticks(rotation=75)
plt.title("PM2.5 distribution by the 20 most represented stations")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "pm25_distribution_by_station.png"), dpi=200, bbox_inches="tight")
plt.show()

## 8. Outlier inspection - IQR, z-scores and Isolation Forest

In [ ]:
outlier_rows = []
for col in eda_vars:
    x = df[col].dropna()
    if len(x) < 5:
        continue
    q1, q3 = x.quantile(.25), x.quantile(.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    z = (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) else pd.Series(0, index=x.index)
    outlier_rows.append({
        "variable": col, "n_observed": len(x), "iqr_lower": low, "iqr_upper": high,
        "iqr_outlier_count": int(((x < low) | (x > high)).sum()),
        "iqr_outlier_percent": float(((x < low) | (x > high)).mean() * 100),
        "abs_zscore_gt_3_count": int((z.abs() > 3).sum())
    })
outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary)
outlier_summary.to_csv(f"{OUT_DIR}/PM25_outlier_summary_IQR_Zscore.csv", index=False)

                                                                                     
if len(eda_vars) >= 2:
    iso_data = df[eda_vars].copy()
    iso_data = iso_data.fillna(iso_data.median(numeric_only=True))
    if len(iso_data) >= 50:
        iso = IsolationForest(contamination="auto", random_state=42, n_estimators=200)
        labels = iso.fit_predict(iso_data)
        df["isolation_forest_flag"] = (labels == -1).astype("int8")
        print("Isolation Forest flagged:", int(df["isolation_forest_flag"].sum()),
              f"({df['isolation_forest_flag'].mean()*100:.2f}%)")
    else:
        print("Isolation Forest skipped: fewer than 50 rows.")
else:
    print("Isolation Forest skipped: insufficient predictors.")

print("Domain rule: retain extreme but plausible PM2.5 pollution episodes. Investigate flagged rows against source metadata; do not delete automatically.")

## 9. Target distribution, high-concentration proportions and seasonal variation

In [ ]:
target = df["pm25"]
target_summary = pd.DataFrame({
    "n": [target.count()],
    "min": [target.min()],
    "max": [target.max()],
    "mean": [target.mean()],
    "median": [target.median()],
    "std": [target.std()],
    "skewness": [target.skew()],
    "kurtosis_excess": [target.kurtosis()],
    "pct_above_25": [(target > 25).mean() * 100],
    "pct_above_50": [(target > 50).mean() * 100]
})
display(target_summary)
target_summary.to_csv(f"{OUT_DIR}/PM25_target_distribution.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(target, bins=50, edgecolor="black")
ax.set_title("PM2.5 target distribution")
ax.set_xlabel("PM2.5 (µg/m³)")
ax.set_ylabel("Count")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "pm25_target_distribution.png"), dpi=200, bbox_inches="tight")
plt.show()

monthly = df.assign(month=df.date.dt.month).groupby("month").agg(
    n=("pm25", "size"),
    median=("pm25", "median"),
    q1=("pm25", lambda x: x.quantile(.25)),
    q3=("pm25", lambda x: x.quantile(.75)),
    pct_above_25=("pm25", lambda x: (x > 25).mean() * 100),
    pct_above_50=("pm25", lambda x: (x > 50).mean() * 100)
).reset_index()
display(monthly)
monthly.to_csv(f"{OUT_DIR}/PM25_monthly_summary.csv", index=False)

plt.figure(figsize=(9, 4))
plt.plot(monthly.month, monthly["median"], marker="o", label="Median")
plt.fill_between(monthly.month, monthly.q1, monthly.q3, alpha=.2, label="IQR")
plt.xticks(range(1, 13))
plt.xlabel("Month")
plt.ylabel("PM2.5 (µg/m³)")
plt.title("Monthly PM2.5 pattern (all years)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "monthly_pm25_pattern.png"), dpi=200, bbox_inches="tight")
plt.show()

yearly = df.assign(year=df.date.dt.year).groupby("year").agg(
    n=("pm25", "size"), median=("pm25", "median"),
    q1=("pm25", lambda x: x.quantile(.25)),
    q3=("pm25", lambda x: x.quantile(.75)),
    pct_above_25=("pm25", lambda x: (x > 25).mean() * 100),
    pct_above_50=("pm25", lambda x: (x > 50).mean() * 100)
).reset_index()
display(yearly)
yearly.to_csv(f"{OUT_DIR}/PM25_yearly_summary.csv", index=False)

## 10. Pearson/Spearman correlations, highly correlated predictors and VIF

In [ ]:
corr_vars = [c for c in eda_vars if c in df.columns]
pearson_corr = df[corr_vars].corr(method="pearson")
spearman_corr = df[corr_vars].corr(method="spearman")
display(pearson_corr)
display(spearman_corr)
pearson_corr.to_csv(f"{OUT_DIR}/PM25_Pearson_Correlation.csv")
spearman_corr.to_csv(f"{OUT_DIR}/PM25_Spearman_Correlation.csv")

plt.figure(figsize=(10, 8))
sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Pearson correlation matrix")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "pearson_correlation_heatmap.png"), dpi=200, bbox_inches="tight")
plt.show()

predictors_for_corr = [c for c in corr_vars if c != "pm25"]
high_corr_pairs = []
for i, a in enumerate(predictors_for_corr):
    for b in predictors_for_corr[i+1:]:
        r = pearson_corr.loc[a, b]
        if pd.notna(r) and abs(r) > .9:
            high_corr_pairs.append({"variable_1": a, "variable_2": b, "pearson_r": r})
high_corr = pd.DataFrame(high_corr_pairs, columns=["variable_1", "variable_2", "pearson_r"])
display(high_corr)
high_corr.to_csv(f"{OUT_DIR}/PM25_highly_correlated_predictors_abs_r_gt_0_9.csv", index=False)

                                                                                     
vif_cols = [c for c in predictors_for_corr if df[c].notna().sum() > 0]
vif_data = df[vif_cols].dropna()
if len(vif_cols) >= 2 and len(vif_data) > len(vif_cols) + 2:
    varying = [c for c in vif_cols if vif_data[c].nunique() > 1]
    vif_data = vif_data[varying]
    if len(varying) >= 2:
        vif_table = pd.DataFrame({
            "variable": varying,
            "VIF": [variance_inflation_factor(vif_data.to_numpy(dtype=float), i)
                    for i in range(len(varying))]
        })
        display(vif_table)
        vif_table.to_csv(f"{OUT_DIR}/PM25_VIF.csv", index=False)
    else:
        print("VIF skipped: fewer than two non-constant complete predictors.")
else:
    print("VIF skipped: insufficient complete-case observations. Correlation analysis remains available.")

## 11. PM2.5 relationships by station/season and meteorological influence on PM2.5-AOD

In [ ]:
                                                                                              
relationship_vars = [c for c in ["AOD", "NO2", "CO", "O3", "temperature_celsius",
                                  "pressure_mb", "wind_u", "wind_v"] if c in df.columns]
station_rel_rows = []
for sid, g in df.groupby("id"):
    for v in relationship_vars:
        pair = g[["pm25", v]].dropna()
        if len(pair) >= 20 and pair["pm25"].nunique() > 1 and pair[v].nunique() > 1:
            station_rel_rows.append({
                "id": sid, "predictor": v, "n": len(pair),
                "pearson_r": pair.pm25.corr(pair[v], method="pearson"),
                "spearman_r": pair.pm25.corr(pair[v], method="spearman")
            })
station_relationships = pd.DataFrame(station_rel_rows)
display(station_relationships.head(30))
station_relationships.to_csv(f"{OUT_DIR}/PM25_predictor_relationships_by_station.csv", index=False)

                                         
month_to_season = {12:"Winter", 1:"Winter", 2:"Winter",
                   3:"Spring", 4:"Spring", 5:"Spring",
                   6:"Summer", 7:"Summer", 8:"Summer",
                   9:"Autumn", 10:"Autumn", 11:"Autumn"}
df["season"] = df.date.dt.month.map(month_to_season)

season_rows = []
for season, g in df.groupby("season"):
    for v in relationship_vars:
        pair = g[["pm25", v]].dropna()
        if len(pair) >= 20:
            season_rows.append({"season": season, "predictor": v, "n": len(pair),
                                "pearson_r": pair.pm25.corr(pair[v]),
                                "spearman_r": pair.pm25.corr(pair[v], method="spearman")})
season_relationships = pd.DataFrame(season_rows)
display(season_relationships)
season_relationships.to_csv(f"{OUT_DIR}/PM25_predictor_relationships_by_season.csv", index=False)

                                                                                                              
if "AOD" in df.columns:
    meteo_candidates = [c for c in ["temperature_celsius", "wind_u", "wind_v"] if c in df.columns]
    if meteo_candidates:
        meteo = df[["pm25", "AOD"] + meteo_candidates].dropna().copy()
        if len(meteo) >= 40:
            meteo["temperature_regime"] = pd.qcut(
                meteo["temperature_celsius"], q=3, labels=["low", "medium", "high"],
                duplicates="drop"
            ) if "temperature_celsius" in meteo else "all"
            wind_speed = np.sqrt(meteo["wind_u"]**2 + meteo["wind_v"]**2) if {"wind_u","wind_v"}.issubset(meteo.columns) else None
            if wind_speed is not None:
                meteo["wind_regime"] = pd.qcut(wind_speed, q=3, labels=["low", "medium", "high"], duplicates="drop")
            else:
                meteo["wind_regime"] = "not_available"
            meteo_aod = meteo.groupby(["temperature_regime", "wind_regime"], observed=True).apply(
                lambda x: pd.Series({
                    "n": len(x),
                    "pm25_aod_pearson": x.pm25.corr(x.AOD),
                    "pm25_aod_spearman": x.pm25.corr(x.AOD, method="spearman")
                }), include_groups=False
            ).reset_index()
            display(meteo_aod)
            meteo_aod.to_csv(f"{OUT_DIR}/PM25_AOD_meteorological_stratification.csv", index=False)
        else:
            print("PM2.5–AOD meteorological stratification skipped: insufficient complete rows.")
    else:
        print("No meteorological variables available for PM2.5–AOD stratification.")

## 12. Lead-lag exploration (same day and past days)

In [ ]:

lead_lag_rows = []

lead_lag_vars = [
    c for c in [
        "pm25", "AOD", "NO2", "CO", "O3",
        "temperature_celsius", "pressure_mb", "wind_u", "wind_v"
    ]
    if c in df.columns
]

                              
if not df.columns.is_unique:
    raise ValueError("Duplicate column names detected in df.")

for var in lead_lag_vars:

    for lag in [0, 1, 2, 3, 7]:

        station_rs = []

        for sid, g in df.groupby("id"):

            g = g.sort_values("date").copy()

                                                    
            if g["date"].duplicated().any():
                continue

            g = g.set_index("date")

                                              
            target_series = g["pm25"].asfreq("D")
            predictor_series = g[var].asfreq("D")

                                                             
            if var == "pm25" and lag == 0:
                continue

                                    
            predictor_lagged = predictor_series.shift(lag)

                                    
            pair = pd.concat(
                [
                    target_series.rename("target"),
                    predictor_lagged.rename("predictor")
                ],
                axis=1
            ).dropna()

                                                                  
            if (
                len(pair) >= 15
                and pair["target"].nunique() > 1
                and pair["predictor"].nunique() > 1
            ):
                correlation = pair["target"].corr(
                    pair["predictor"], method="pearson"
                )

                station_rs.append(correlation)

        lead_lag_rows.append({
            "predictor": var,
            "lag_days": lag,
            "stations_with_estimate": len(station_rs),
            "median_station_pearson_r":
                np.median(station_rs) if station_rs else np.nan
        })

                      
lead_lag = pd.DataFrame(lead_lag_rows)

display(lead_lag)

              
lead_lag.to_csv(
    f"{OUT_DIR}/PM25_lead_lag_correlations.csv",
    index=False
)

               
plt.figure(figsize=(10, 5))

for var, g in lead_lag.groupby("predictor"):

    plt.plot(
        g["lag_days"],
        g["median_station_pearson_r"],
        marker="o",
        label=var
    )

plt.axhline(0, color="black", linewidth=0.8)

plt.xticks([0, 1, 2, 3, 7])
plt.xlabel("Predictor lag (days)")
plt.ylabel("Median station Pearson correlation")
plt.title("Exploratory lead–lag association (lag 0 is not a model input)")

plt.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "lead_lag_correlations.png"), dpi=200, bbox_inches="tight")
plt.show()
